In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier

import optuna

import shap

import category_encoders as ce

import joblib

import warnings
warnings.filterwarnings('ignore')

In [ ]:
class KingaMetricCreditRiskModel:
    def __init__(self):
        self.model = None
        self.scaler = StandardScaler()
        self.label_encoders = {}
        self.feature_names = None
        self.leaky_features = ['Payment_Behaviour', 'Delay_from_due_date', 'Num_of_Delayed_Payment']
    
    def load_and_preprocess(self, filepath):
        df = pd.read_csv(filepath)

        #print(f'Dataset shape: {df.shape}, Default rate: {df["Default_Flag"].mean():.3f}')
        
        
        # Drop leaky features (future/default indicators)
        df = df.drop(columns=[c for c in self.leaky_features if c in df.columns], errors='ignore')

        print('Dropped leaky features:', [c for c in self.leaky_features if c in df])

        df = self.add_interaction_features(df) 

        # Outlier clipping + fillna
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        """for col in df.select_dtypes('number').columns:
            Q1, Q3 = df[col].quantile([0.25, 0.75])
            IQR = Q3 - Q1
            df[col] = df[col].clip(lower=Q1-1.5*IQR, upper=Q3+1.5*IQR)"""

        df.fillna(df.median(numeric_only=True), inplace=True)
        
        X = df.drop('Default_Flag', axis=1)
        y = df['Default_Flag']

        X = self.target_encode(X, y)

        self.feature_names = X.columns.tolist()
        
        return X, y
    
    def target_encode(self, X, y):

        cat_cols = [c for c in ["Payment_of_Min_Amount", "Credit_Mix", "Borrower_Tier"] if c in X.columns]

        encoder = ce.TargetEncoder(cols=cat_cols, smoothing=10, min_samples_leaf=50)
        X_encoded = encoder.fit_transform(X, y)

        self.target_encoder = encoder

        return X_encoded
    
    def add_interaction_features(self, df):
        df = df.copy()

        df["Debt_Stress"] = df["normalized_dti"] * df["normalized_utilization"]
        df["Repayment_Stress"] = df["normalized_emi"] * df["normalized_delinquency"]
        df["Liquidity_Index"] = df["normalized_savings"] * df["normalized_emi"]
        df["Credit_Exposure"] = df["Num_Credit_Card"] * df["Credit_Utilization_Ratio"]

        return df
    
    def feature_selection(self, X, y):

        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        importances = pd.Series(0, index=X.columns)

        for train_idx, val_idx in folds.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]

            model = XGBClassifier(n_estimators=300, max_depth=4, random_state=42)
            model.fit(X_train, y_train)

            importances += pd.Series(model.feature_importances_, index=X.columns)
        
        importances /= folds.n_splits 

        top_features = importances.sort_values(ascending=False).head(15).index

        print("Top Features: ", top_features.tolist())

        return X[top_features], top_features
    
    def objective(self, trial, X_temp, y_temp, folds):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 300, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 5),
            'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.08),
            'subsample': trial.suggest_float('subsample', 0.7, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
            'min_child_weight': trial.suggest_int('min_child_weight', 5, 15),
            'gamma': trial.suggest_float('gamma', 0.1, 0.5),
            'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 1),
            'reg_lambda': trial.suggest_float('reg_lambda', 1, 5),
            'random_state': 42,
            'tree_method': 'hist',
            'eval_metric': 'auc'
        }
        
        auc_scores = []

        for train_idx, val_idx in folds.split(X_temp, y_temp):
            X_train, X_val = X_temp.iloc[train_idx], X_temp.iloc[val_idx]
            y_train, y_val = y_temp.iloc[train_idx], y_temp.iloc[val_idx]
            
            # SMOTE oversample train
            #smote = SMOTE(random_state=42)
            #X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

            pos = (y_temp ==1).sum()
            neg = (y_temp ==0).sum()

            scale_pos_weight = neg/ (pos + 1e-6)
            params['scale_pos_weight'] = scale_pos_weight
            
            model = XGBClassifier(**params)
            model.fit(X_train, y_train)
            y_pred = model.predict_proba(X_val)[:,1]
            auc_scores.append(roc_auc_score(y_val, y_pred))
        
        return np.mean(auc_scores)
    
    def tune_hyperparams(self, X_temp, y_temp):
        folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner(n_startup_trials=10))
        
        study.optimize(lambda trial: self.objective(trial, X_temp, y_temp, folds), n_trials=200)
        
        return study.best_params
    
    def optimize_threshold(self, y_ture, y_probs):

        thresholds = np.linspace(0.1, 0.9, 50)
        best_ks = 0
        best_thresh = 0.5

        for t in thresholds:
            preds = (y_probs >= t).astype(int)

            ks = shap.ks_2samp(
                y_probs[y_ture == 0],
                y_probs[y_ture == 1]
            ).statistic

            if ks > best_ks:
                best_ks = ks
                best_thresh = t 

        self.best_threshold = best_thresh

        return best_thresh
    
    def train(self, filepath):
        print('Loading data...')
        X, y = self.load_and_preprocess(filepath)
        
        print('Feature selection...')
        X_selected, self.feature_names = self.feature_selection(X, y)
        
        print('Splitting...')
        X_temp, X_test, y_temp, y_test = train_test_split(X_selected, y, test_size=0.2, stratify=y, random_state=42)
        
        print('Tuning hyperparams...')
        best_params = self.tune_hyperparams(X_temp, y_temp)
        print('Best params:', best_params)

        X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.2, stratify=y_temp, random_state=42)

        self.model = XGBClassifier(**best_params, random_state=42)

        self.model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)

        y_probs = self.model.predict_proba(X_val)[:,1]
        best_thresh = self.optimize_threshold(y_val, y_probs)

        print(f"OPtimal Threshold: {best_thresh:.3f}")        
        test_auc = roc_auc_score(y_test, self.model.predict_proba(X_test)[:,1])
        
        print(f'Test AUC: {test_auc:.4f}')
        
        if test_auc >= 0.85:
            print('🎉 Target AUC 0.85 achieved!')
        else:
            print('Target not met, consider ensemble/more features.')
        
        return test_auc
    
    def save(self, path='kinga_model.pkl'):
        joblib.dump(self, path)
        
        print(f'Model saved to {path}')

    def predict_proba(self, X_new):
        X_new = self.add_interaction_features(X_new)

        X_new = self.target_encoder.transform(X_new)

        X_new = X_new[self.feature_names]
        
        return self.model.predict_proba(X_new)[:,1]
        

if __name__ == '__main__':
    model = KingaMetricCreditRiskModel()
    auc = model.train('datasets/kingametric_credit_risk.csv')
    
    model.save()

Loading data...
Dropped leaky features: []
Feature selection...


[I 2026-03-23 13:36:54,498] A new study created in memory with name: no-name-e4313a32-0b0b-443a-aa69-fd1a0ea901b7


Top Features:  ['Borrower_Tier', 'normalized_delinquency', 'Payment_of_Min_Amount', 'Payment_Instability', 'population_density_factor', 'Num_of_Loan', 'normalized_inquiry_intensity', 'Repayment_Stress', 'Credit_Depth', 'Credit_Instability', 'Changed_Credit_Limit', 'normalized_savings_capacity_ratio', 'Total_EMI_per_month', 'Outstanding_Debt', 'normalized_dti']
Splitting...
Tuning hyperparams...


[I 2026-03-23 13:36:56,676] Trial 0 finished with value: 0.609936730260528 and parameters: {'n_estimators': 766, 'max_depth': 5, 'learning_rate': 0.056990349806220725, 'subsample': 0.8624389725482089, 'colsample_bytree': 0.7144626059641895, 'min_child_weight': 10, 'gamma': 0.30724853140005737, 'reg_alpha': 0.546881455635446, 'reg_lambda': 1.07726406733841}. Best is trial 0 with value: 0.609936730260528.
[I 2026-03-23 13:36:58,367] Trial 1 finished with value: 0.6144761026228 and parameters: {'n_estimators': 739, 'max_depth': 4, 'learning_rate': 0.05808780172297186, 'subsample': 0.8998491319091095, 'colsample_bytree': 0.8509522532689526, 'min_child_weight': 13, 'gamma': 0.42906755656844164, 'reg_alpha': 0.8491358188013008, 'reg_lambda': 1.0793147914012167}. Best is trial 1 with value: 0.6144761026228.
[I 2026-03-23 13:36:59,726] Trial 2 finished with value: 0.6177447913536632 and parameters: {'n_estimators': 696, 'max_depth': 3, 'learning_rate': 0.07036117426879028, 'subsample': 0.86580